# U-ADAPT — Supervisor Demo Visualizations

**Mode A uncertainty-gated fusion — publication-quality evidence of the research contribution.**

This notebook renders the six demo figures from the outputs of
`scripts/demo_mode_a_end_to_end.py`. If the results do not exist yet, the
first cell runs the demo pipeline automatically (deterministic, seed=0,
synthetic world or real cache).

| # | Figure | What it proves |
|---|--------|----------------|
| 1 | Gate weight distribution | the gate is **dynamic** (not stuck at w=0.5) |
| 2 | Uncertainty–accuracy (D1/D2) | uncertainty proxies **predict error** (core assumption) |
| 3 | Gate favorability (D3) | the gate **chooses the better modality** > 50% |
| 4 | Gap recovery | U-ADAPT **recovers part of the zero-shot→transfer gap** |
| 5 | Qualitative examples | gate behavior on real (or synthetic) proposals |
| 6 | Coefficient ablation | each gate component (alpha/beta/gamma) contributes |

**Validation checklist:** deterministic (seed=0) · figures render · runs in
<30 min on Colab · supervisor-ready.

In [ ]:
# --- setup: run demo if results missing, then load ------------------------
import json
import os
import subprocess
import sys
from pathlib import Path

# locate repo root: walk up from cwd and check known Colab mount paths
def _find_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/u-adapt-disaster-perception")]
    for cand in candidates:
        if (cand / "scripts" / "demo_mode_a_end_to_end.py").exists():
            return cand
    return Path.cwd()

ROOT = _find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

OUT = ROOT / "outputs" / "supervisor_demo"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
RESULTS = OUT / "results.json"
PROPOSALS = OUT / "proposal_level.json"

if not RESULTS.exists():
    print("results.json not found — running demo pipeline (seed=0)...")
    subprocess.run(
        [sys.executable, "scripts/demo_mode_a_end_to_end.py",
         "--out", str(RESULTS), "--proposal-out", str(PROPOSALS)],
        cwd=ROOT, check=True,
    )

results = json.loads(RESULTS.read_text())
proposal_payload = json.loads(PROPOSALS.read_text())
proposals = proposal_payload["proposals"]
ground_truth = proposal_payload["ground_truth"]

import matplotlib.pyplot as plt
import numpy as np
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    pass  # seaborn optional; plain matplotlib styling is fine

from uadapt.demo.plotting import (
    figure1_gate_weights,
    figure2_d1_d2,
    figure3_gate_favorability,
    figure4_gap_recovery,
    figure5_qualitative,
    figure6_ablation,
)

print("data source:", results["meta"].get("data_source"))
print("mAP50:", {k: round(v, 4) for k, v in results["map50"].items()})

## Figure 1 — Gate Weight Distribution

**Claim:** the gate is *dynamic* — it does not collapse to naive averaging
(w = 0.5). A wide spread of w proves the analytic rule is responding to
per-proposal uncertainty (σ²_text, σ²_visual, affinity). Color shows which
modality was correct: if the gate works, text-correct proposals should skew
toward low w and visual-correct proposals toward high w.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
note = figure1_gate_weights(proposals, ax)
fig.tight_layout()
fig.savefig(FIG / "figure1_gate_weights.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)

## Figure 2 — Uncertainty–Accuracy Correlation (D1/D2)

**Claim (core assumption):** higher normalized uncertainty ⇒ higher error
rate. Binned scatter of error rate vs σ²_text (D1) and σ²_visual (D2) with
Spearman ρ — the *validity* of the whole gating mechanism rests on these.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
note = figure2_d1_d2(results["diagnostics"], axes)
fig.tight_layout()
fig.savefig(FIG / "figure2_d1_d2.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)

## Figure 3 — Gate Favorability (D3)

**Claim (is the gate useful?):** among proposals where the two modalities
*disagree*, the gate assigns higher weight to the more accurate modality
significantly more often than chance (binomial test vs 50%).

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
note = figure3_gate_favorability(results["diagnostics"], ax)
fig.tight_layout()
fig.savefig(FIG / "figure3_gate_favorability.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)

## Figure 4 — Gap Recovery Analysis

**Claim (RQ2):** U-ADAPT recovers a fraction of the zero-shot→transfer gap:

$$\text{gap recovery} = \frac{mAP_{U-ADAPT} - mAP_{zero}}{mAP_{oracle} - mAP_{zero}} \times 100\%$$

Bars: zero-shot (raw scores) → U-ADAPT Mode A → transfer ceiling (oracle
re-rank: every GT-correct proposal ranked above every incorrect one — the
maximum any re-scoring method can reach on this proposal set). Dashed lines
(if configured) show the literature zero-shot / transfer mAP50 from the
dataset config — **dataset-specific references (e.g. D-Fire 27.5 → 65.6),
not a comparable baseline** to the demo subset.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
note = figure4_gap_recovery(results["gap_recovery"], ax)
fig.tight_layout()
fig.savefig(FIG / "figure4_gap_recovery.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)
print("gap recovery payload:", {k: round(v, 4) if isinstance(v, float) else v
                                 for k, v in results["gap_recovery"].items()})

## Figure 5 — Qualitative Examples

Three panels (schematic until real imagery is available):

1. **High w** — the gate trusted *visual* (low visual uncertainty / high affinity);
2. **Low w** — the gate trusted *text* (high visual uncertainty / low affinity);
3. **Gate corrected naive averaging** — the two modalities disagreed and the
   gate picked the correct one, where w=0.5 averaging would have diluted it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
note = figure5_qualitative(proposals, ground_truth, axes, seed=0)
fig.tight_layout()
fig.savefig(FIG / "figure5_qualitative.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)

## Figure 6 — Ablation Study

**Claim (component contributions):** removing a term changes gate behavior.
On the demo subset, **alpha (visual uncertainty) contributes** (alpha=0 drops
mAP50); beta (text uncertainty) and gamma (affinity) effects are small /
within noise — expected on this synthetic world, where the gating signal is
dominated by the visual-uncertainty term. The figure reports the numbers
honestly; significance testing is deferred to the real-data protocol (§9).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
note = figure6_ablation(results["ablation"], ax)
fig.tight_layout()
fig.savefig(FIG / "figure6_ablation.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)

## Summary & Next Steps

Figures are saved to `outputs/supervisor_demo/figures/`. The 2-page
supervisor report is at `docs/supervisor_demo_report.md` (embeds these
figures and interprets the D1–D3 diagnostics).

**Next steps before thesis submission:**
- Re-run on real cached features (Milestone 1+): `--cache-dir cached_features
  --ground-truth data/annotations/dfire_test.json`;
- 10-seed statistical protocol (paired t-test + Wilcoxon, §9);
- full mAP50:95 + ECE/Brier/uncertainty-AUROC via `scripts/04_evaluate.py`;
- cross-backbone ablation (OWL-ViT, YOLOE26) for RQ5.

*Synthetic-data caveat: the numbers above are a mechanism demonstration.
They validate the wiring and the diagnostics — not a research result.*